# Colab에서 응집도 스케일 검증 실험 실행

GitHub 저장소에서 직접 클론하여 실행합니다.
- 저장소: `jack0682/Perception_theory`
- 실험: `exp_cohesion_scale.py` (병렬 처리)

## 셀 1️⃣: GitHub 저장소 클론

In [ ]:
# GitHub에서 저장소 클론
!git clone https://github.com/jack0682/Perception_theory.git /content/Perception_theory

# 작업 디렉토리 설정
%cd /content/Perception_theory

# 디렉토리 확인
import os
print("\n✓ 클론 완료")
print(f"현재 디렉토리: {os.getcwd()}")
print(f"\n주요 폴더:")
for item in os.listdir('.'):
    if os.path.isdir(item) and not item.startswith('.'):
        print(f"  - {item}/")

## 셀 2️⃣: 의존성 설치

In [ ]:
# 필요한 패키지 설치
!pip install -q numpy scipy networkx matplotlib

print("✓ 의존성 설치 완료")

## 셀 3️⃣: Python 경로 설정 및 임포트 확인

In [ ]:
import sys
sys.path.insert(0, '/content/Perception_theory')

# 모듈 임포트 확인
try:
    from scc.graph import GraphState
    from scc.params import ParameterRegistry
    from scc.optimizer import find_formation
    print("✓ SCC 모듈 임포트 성공")
except ImportError as e:
    print(f"✗ 임포트 실패: {e}")

## 셀 4️⃣: 실험 실행 (병렬 처리)

In [ ]:
# 실험 파라미터
max_size = 256          # 최대 격자 크기
c_ref = 0.25            # 부피 분수
n_workers = 2           # Colab은 2-4 권장
output_file = 'experiments/results/cohesion_scale_test.json'

print(f"실험 파라미터:")
print(f"  max_size: {max_size}")
print(f"  c_ref: {c_ref}")
print(f"  n_workers: {n_workers}")
print(f"  output: {output_file}")
print()

In [ ]:
# 실험 실행
import subprocess

cmd = [
    'python3',
    'experiments/exp_cohesion_scale.py',
    '--max-size', str(max_size),
    '--c-ref', str(c_ref),
    '--n-workers', str(n_workers),
    '--output', output_file
]

print(f"실행: {' '.join(cmd)}")
print("="*130)
print()

result = subprocess.run(cmd, capture_output=True, text=True)

# 출력 표시
if result.stdout:
    print(result.stdout)
if result.stderr:
    print("[STDERR]")
    print(result.stderr)

if result.returncode == 0:
    print("\n✓ 실험 완료")
else:
    print(f"\n✗ 오류 발생 (return code: {result.returncode})")

## 셀 5️⃣: 결과 로드 및 분석

In [ ]:
import json
import numpy as np

# 결과 로드
with open(output_file) as f:
    results = json.load(f)

print(f"\n{'='*130}")
print("분석")
print(f"{'='*130}\n")

sorted_keys = sorted(results.keys())
bind_vals = [results[k]['bind'] for k in sorted_keys]
sep_vals = [results[k]['sep'] for k in sorted_keys]
time_vals = [results[k]['time_sec'] for k in sorted_keys]

# 테이블
print(f"{'Grid':>10} {'Bind':>10} {'Sep':>10} {'Inside':>10} {'Persist':>10} {'Energy':>12} {'Time(s)':>10} {'Iter':>8}")
print("-"*130)
for k in sorted_keys:
    r = results[k]
    print(f"{k:>10} {r['bind']:>10.4f} {r['sep']:>10.4f} {r['inside']:>10.4f} {r['persist']:>10.4f} "
          f"{r['energy']:>12.6f} {r['time_sec']:>10.1f} {r['n_iter']:>8d}")

# 통계
print(f"\n{'─'*130}")
print("통계:")
print(f"  Bind:  min={min(bind_vals):.4f}, max={max(bind_vals):.4f}, "
      f"mean={np.mean(bind_vals):.4f}, std={np.std(bind_vals):.4f}")
print(f"  Sep:   min={min(sep_vals):.4f}, max={max(sep_vals):.4f}, "
      f"mean={np.mean(sep_vals):.4f}, std={np.std(sep_vals):.4f}")
print(f"  Time:  total={sum(time_vals):.1f}s, mean={np.mean(time_vals):.1f}s/grid")

# 결론
print(f"\n{'─'*130}")
print("결론:")

if min(bind_vals) > 0.5:
    print(f"  ✓ 강한 응집: 모든 격자에서 Bind > 0.5")
elif min(bind_vals) > 0.3:
    print(f"  △ 중간 응집: 모든 격자에서 0.3 < Bind < 0.5")
else:
    print(f"  ✗ 약한 응집: Bind < 0.3")

bind_change = (bind_vals[-1] - bind_vals[0]) / bind_vals[0] * 100 if bind_vals[0] != 0 else 0
if abs(bind_change) < 10:
    print(f"  ✓ 안정적: Bind 변화 {bind_change:+.1f}% (< 10%)")
else:
    print(f"  △ 변동: Bind 변화 {bind_change:+.1f}%")

if all(sep < 0.5 for sep in sep_vals):
    print(f"  ✓ 분리 안정: 모든 격자에서 Sep < 0.5")
else:
    print(f"  △ 분리 증가: 일부 격자에서 Sep > 0.5")

print(f"\n{'='*130}")

## 셀 6️⃣: 시각화

In [ ]:
import matplotlib.pyplot as plt

if len(results) >= 2:
    sorted_keys = sorted(results.keys())
    grid_labels = [k.replace('x', '×') for k in sorted_keys]
    
    bind_vals = [results[k]['bind'] for k in sorted_keys]
    sep_vals = [results[k]['sep'] for k in sorted_keys]
    inside_vals = [results[k]['inside'] for k in sorted_keys]
    persist_vals = [results[k]['persist'] for k in sorted_keys]
    energy_vals = [results[k]['energy'] for k in sorted_keys]
    time_vals = [results[k]['time_sec'] for k in sorted_keys]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(f'응집도 스케일 검증 (c_ref={c_ref})', fontsize=16, fontweight='bold')

    # Bind
    axes[0, 0].plot(grid_labels, bind_vals, 'o-', color='#2E86AB', linewidth=2, markersize=8)
    axes[0, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold=0.5')
    axes[0, 0].set_title('Bind (응집 강도)', fontweight='bold')
    axes[0, 0].set_ylabel('Bind')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()

    # Sep
    axes[0, 1].plot(grid_labels, sep_vals, 's-', color='#A23B72', linewidth=2, markersize=8)
    axes[0, 1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold=0.5')
    axes[0, 1].set_title('Sep (분리도)', fontweight='bold')
    axes[0, 1].set_ylabel('Sep')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()

    # Inside
    axes[0, 2].plot(grid_labels, inside_vals, '^-', color='#F18F01', linewidth=2, markersize=8)
    axes[0, 2].set_title('Inside (내부성)', fontweight='bold')
    axes[0, 2].set_ylabel('Inside')
    axes[0, 2].grid(True, alpha=0.3)

    # Persist
    axes[1, 0].plot(grid_labels, persist_vals, 'd-', color='#06A77D', linewidth=2, markersize=8)
    axes[1, 0].set_title('Persist (지속성)', fontweight='bold')
    axes[1, 0].set_ylabel('Persist')
    axes[1, 0].grid(True, alpha=0.3)

    # Energy
    axes[1, 1].plot(grid_labels, energy_vals, 'p-', color='#D62828', linewidth=2, markersize=8)
    axes[1, 1].set_title('Energy (전체 에너지)', fontweight='bold')
    axes[1, 1].set_ylabel('Energy')
    axes[1, 1].grid(True, alpha=0.3)

    # Time
    axes[1, 2].bar(grid_labels, time_vals, color='#1B998B', alpha=0.7, edgecolor='black')
    axes[1, 2].set_title('실행 시간', fontweight='bold')
    axes[1, 2].set_ylabel('Time (s)')
    axes[1, 2].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('experiments/results/cohesion_scale_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ 그래프 저장: experiments/results/cohesion_scale_analysis.png")
else:
    print("그래프를 그리기 위해 2개 이상의 결과가 필요합니다.")

## 셀 7️⃣: 결과 다운로드

In [ ]:
from google.colab import files

print("Colab에서 로컬으로 파일을 다운로드합니다...\n")

# JSON 결과 다운로드
try:
    files.download(output_file)
    print(f"✓ {output_file} 다운로드")
except FileNotFoundError:
    print(f"✗ {output_file} 없음")

# 그래프 다운로드
try:
    files.download('experiments/results/cohesion_scale_analysis.png')
    print(f"✓ cohesion_scale_analysis.png 다운로드")
except FileNotFoundError:
    print(f"✗ cohesion_scale_analysis.png 없음")

print("\n✓ 모든 파일 준비 완료")

## 셀 8️⃣: 최종 요약

In [ ]:
from datetime import datetime

print("\n" + "="*130)
print("Colab 실험 완료")
print("="*130)
print(f"시간: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\n결과 파일:")
print(f"  - {output_file}")
print(f"  - experiments/results/cohesion_scale_analysis.png")
print(f"\n처리된 격자:")
for k in sorted(results.keys()):
    print(f"  ✓ {k}: {results[k]['time_sec']:.1f}s")
print(f"\n총 시간: {sum([results[k]['time_sec'] for k in results.keys()]):.1f}s")
print("="*130)